# Load Instance Type Pricing from Excel

This notebook loads the `Instance Type Pricing.xlsx` file and creates the `instance_rates` table.

**Source:** `Instance Type Pricing.xlsx` (in same folder as this notebook)  
**Target:** `lakemeter_catalog.lakemeter.instance_rates`


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_INSTANCE_RATES = f"{CATALOG}.{SCHEMA}.instance_rates"

# Excel file is in the same workspace folder as this notebook
import os
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = "/Workspace" + "/".join(notebook_path.split("/")[:-1])
INSTANCE_PRICING_EXCEL = f"{notebook_dir}/Instance Type Pricing.xlsx"

print(f"✅ Config: {CATALOG}.{SCHEMA}")
print(f"📄 Excel: {INSTANCE_PRICING_EXCEL}")


In [ ]:
# Install required packages
%pip install pandas openpyxl -q


In [ ]:
# Read Excel file and explore sheets
import pandas as pd

# Read all sheets
excel_file = pd.ExcelFile(INSTANCE_PRICING_EXCEL)
print(f"📄 File: {INSTANCE_PRICING_EXCEL}")
print(f"📑 Sheets: {excel_file.sheet_names}")


In [ ]:
# Load AWS sheet
df_aws_raw = pd.read_excel(INSTANCE_PRICING_EXCEL, sheet_name='AWS', header=2)
print(f"📊 AWS Sheet Columns: {list(df_aws_raw.columns)}")
print(f"📊 AWS Sheet Rows: {len(df_aws_raw)}")
df_aws_raw.head(10)


In [ ]:
# =============================================================================
# HARDCODED INSTANCE FAMILY MAPPING BY CLOUD
# =============================================================================

def get_aws_family(instance_type):
    """Map AWS instance type to standard family based on prefix."""
    if pd.isna(instance_type):
        return None
    
    inst = str(instance_type).lower()
    prefix = inst.split('.')[0] if '.' in inst else inst
    
    # GPU instances (p, g, inf, trn)
    if prefix.startswith(('p2', 'p3', 'p4', 'p5', 'g3', 'g4', 'g5', 'g6', 'inf', 'trn')):
        return 'GPU'
    
    # Storage Optimized (i, d, h)
    if prefix.startswith(('i3', 'i4', 'd2', 'd3', 'h1')):
        return 'Storage Optimized'
    
    # Memory Optimized (r, x, z)
    if prefix.startswith(('r3', 'r4', 'r5', 'r6', 'r7', 'x1', 'x2', 'z1')):
        return 'Memory Optimized'
    
    # Compute Optimized (c)
    if prefix.startswith(('c3', 'c4', 'c5', 'c6', 'c7')):
        return 'Compute Optimized'
    
    # Burstable (t)
    if prefix.startswith(('t2', 't3', 't4')):
        return 'Burstable'
    
    # General Purpose (m, a - default)
    return 'General Purpose'


def get_azure_family(instance_type):
    """Map Azure instance type to standard family based on prefix."""
    if pd.isna(instance_type):
        return None
    
    inst = str(instance_type).upper().replace('STANDARD_', '')
    
    # GPU instances (NC, ND, NV, NP)
    if any(inst.startswith(x) for x in ['NC', 'ND', 'NV', 'NP']):
        return 'GPU'
    
    # Storage Optimized (L series)
    if inst.startswith('L'):
        return 'Storage Optimized'
    
    # Memory Optimized (E, M series)
    if inst.startswith(('E', 'M')):
        return 'Memory Optimized'
    
    # Compute Optimized (F series)
    if inst.startswith('F'):
        return 'Compute Optimized'
    
    # Burstable (B series)
    if inst.startswith('B'):
        return 'Burstable'
    
    # General Purpose (D, DS, A, DC series - default)
    return 'General Purpose'


def get_gcp_family(instance_type):
    """Map GCP instance type to standard family based on prefix."""
    if pd.isna(instance_type):
        return None
    
    inst = str(instance_type).lower()
    prefix = inst.split('-')[0] if '-' in inst else inst
    
    # GPU instances (a2, g2)
    if prefix in ['a2', 'g2']:
        return 'GPU'
    
    # Memory Optimized (m1, m2, m3)
    if prefix in ['m1', 'm2', 'm3']:
        return 'Memory Optimized'
    
    # Compute Optimized (c2, c2d, c3)
    if prefix in ['c2', 'c2d', 'c3']:
        return 'Compute Optimized'
    
    # General Purpose (n1, n2, n2d, e2, t2d - default)
    return 'General Purpose'


print("✅ Defined hardcoded instance family mapping functions")


In [ ]:
# Process AWS data with hardcoded family mapping
# AWS structure:
#   Col 3 = Instance Name (m4.large, m5.xlarge) -> instance_type
#   Col 4 = vCPU
#   Col 5 = Memory (GB)
#   Col 7 = DBU Rate

def clean_aws_instance_type(instance_type):
    """
    Clean AWS instance type name.
    - Strip "(Beta)" suffix (AWS Pricing API doesn't include it)
    - Trim whitespace
    """
    if pd.isna(instance_type):
        return None
    inst = str(instance_type).strip()
    # Remove "(Beta)" suffix for easier joining with AWS Pricing API
    inst = inst.replace(' (Beta)', '').replace('(Beta)', '').strip()
    return inst

def process_aws_sheet(df_raw):
    """Process AWS sheet with hardcoded family mapping."""
    df = df_raw.copy()
    
    df_clean = pd.DataFrame()
    df_clean['instance_type_raw'] = df.iloc[:, 3]
    df_clean['vcpus'] = pd.to_numeric(df.iloc[:, 4], errors='coerce')
    df_clean['memory_gb'] = pd.to_numeric(df.iloc[:, 5], errors='coerce')
    df_clean['dbu_rate'] = pd.to_numeric(df.iloc[:, 7], errors='coerce')
    df_clean['cloud'] = 'AWS'
    
    # Filter: keep only rows with valid instance_type (contains .) and dbu_rate
    df_clean = df_clean.dropna(subset=['instance_type_raw', 'dbu_rate'])
    df_clean = df_clean[df_clean['instance_type_raw'].astype(str).str.contains(r'\.', na=False)]
    
    # Clean instance type (strip "(Beta)" suffix)
    df_clean['instance_type'] = df_clean['instance_type_raw'].apply(clean_aws_instance_type)
    
    # Apply hardcoded family mapping based on instance_type prefix
    df_clean['instance_family'] = df_clean['instance_type'].apply(get_aws_family)
    
    # Reorder columns (drop raw column)
    df_clean = df_clean[['cloud', 'instance_type', 'instance_family', 'vcpus', 'memory_gb', 'dbu_rate']]
    
    return df_clean

df_aws = process_aws_sheet(df_aws_raw)
print(f"✅ Processed {len(df_aws)} AWS instance types")
print(f"   Families: {df_aws['instance_family'].unique().tolist()}")
print(f"\n📊 By family:")
print(df_aws.groupby('instance_family').size())
df_aws.head(10)


In [ ]:
# Load Azure sheet
df_azure_raw = pd.read_excel(INSTANCE_PRICING_EXCEL, sheet_name='Azure', header=2)
print(f"📊 Azure Sheet Columns: {list(df_azure_raw.columns)}")
print(f"📊 Azure Sheet Rows: {len(df_azure_raw)}")
df_azure_raw.head(5)


In [ ]:
# Process Azure data with hardcoded family mapping
# Azure structure:
#   Col 2 = Instance Name (DS1 v2, Standard_D2s_v3) -> instance_type
#   Col 3 = vCPU
#   Col 4 = Memory (GB)
#   Col 5 = Databricks Unit DBU -> dbu_rate

def normalize_azure_instance_type(instance_type):
    """
    Normalize Azure instance type to match Azure Retail Prices API format.
    
    Excel format: "D16a_v4", "D16ads v5", "D12 v2"
    API format:   "Standard_D16a_v4", "Standard_D16ads_v5", "Standard_D12_v2"
    """
    if pd.isna(instance_type):
        return None
    
    inst = str(instance_type).strip()
    
    # Replace spaces with underscores
    inst = inst.replace(' ', '_')
    
    # Add "Standard_" prefix if not present
    if not inst.lower().startswith('standard_'):
        inst = f"Standard_{inst}"
    
    return inst

def process_azure_sheet(df_raw):
    """Process Azure sheet with hardcoded family mapping."""
    df = df_raw.copy()
    
    df_clean = pd.DataFrame()
    df_clean['instance_type_raw'] = df.iloc[:, 2]
    df_clean['vcpus'] = pd.to_numeric(df.iloc[:, 3], errors='coerce')
    df_clean['memory_gb'] = pd.to_numeric(df.iloc[:, 4], errors='coerce')
    df_clean['dbu_rate'] = pd.to_numeric(df.iloc[:, 5], errors='coerce')
    df_clean['cloud'] = 'AZURE'
    
    # Filter: keep only rows with numeric vCPU and dbu_rate (data rows, not headers)
    df_clean = df_clean.dropna(subset=['vcpus', 'dbu_rate'])
    
    # Normalize instance type to match Azure Retail Prices API format
    df_clean['instance_type'] = df_clean['instance_type_raw'].apply(normalize_azure_instance_type)
    
    # Apply hardcoded family mapping based on instance_type prefix
    df_clean['instance_family'] = df_clean['instance_type'].apply(get_azure_family)
    
    # Reorder columns (drop raw column)
    df_clean = df_clean[['cloud', 'instance_type', 'instance_family', 'vcpus', 'memory_gb', 'dbu_rate']]
    
    return df_clean

df_azure = process_azure_sheet(df_azure_raw)
print(f"✅ Processed {len(df_azure)} Azure instance types")
print(f"   Families: {df_azure['instance_family'].unique().tolist()}")
print(f"\n📊 By family:")
print(df_azure.groupby('instance_family').size())
df_azure.head(15)


In [ ]:
# Load GCP sheet
df_gcp_raw = pd.read_excel(INSTANCE_PRICING_EXCEL, sheet_name='GCP', header=2)
print(f"📊 GCP Sheet Columns: {list(df_gcp_raw.columns)}")
print(f"📊 GCP Sheet Rows: {len(df_gcp_raw)}")
df_gcp_raw.head(5)


In [ ]:
# Process GCP data with hardcoded family mapping
# GCP structure:
#   Col 3 = Instance Name (n1-standard-4) -> instance_type
#   Col 4 = vCPU
#   Col 5 = Memory (GB)
#   Col 6 = DBU -> dbu_rate

def process_gcp_sheet(df_raw):
    """Process GCP sheet with hardcoded family mapping."""
    df = df_raw.copy()
    
    df_clean = pd.DataFrame()
    df_clean['instance_type'] = df.iloc[:, 3]        # Column 3: Instance Name
    df_clean['vcpus'] = pd.to_numeric(df.iloc[:, 4], errors='coerce')
    df_clean['memory_gb'] = pd.to_numeric(df.iloc[:, 5], errors='coerce')
    df_clean['dbu_rate'] = pd.to_numeric(df.iloc[:, 6], errors='coerce')
    df_clean['cloud'] = 'GCP'
    
    # Filter: keep only data rows
    df_clean = df_clean.dropna(subset=['instance_type', 'dbu_rate'])
    df_clean = df_clean[df_clean['instance_type'].astype(str).str.match(r'^[a-z]\d', na=False)]
    
    # Apply hardcoded family mapping based on instance_type prefix
    df_clean['instance_family'] = df_clean['instance_type'].apply(get_gcp_family)
    
    # Reorder columns
    df_clean = df_clean[['cloud', 'instance_type', 'instance_family', 'vcpus', 'memory_gb', 'dbu_rate']]
    
    return df_clean

df_gcp = process_gcp_sheet(df_gcp_raw)
print(f"✅ Processed {len(df_gcp)} GCP instance types")
print(f"   Families: {df_gcp['instance_family'].unique().tolist()}")
print(f"\n📊 By family:")
print(df_gcp.groupby('instance_family').size())
df_gcp.head(10)


In [ ]:
# Combine all cloud data
from datetime import datetime

df_all = pd.concat([df_aws, df_azure, df_gcp], ignore_index=True)

# Add metadata columns
df_all['source'] = 'Instance Type Pricing.xlsx'
df_all['is_active'] = True
df_all['updated_at'] = datetime.utcnow()

# DEDUPLICATE: Remove duplicate (cloud, instance_type) combinations
# Keep first occurrence (in case of different dbu_rate values, this logs a warning)
before_count = len(df_all)
duplicates = df_all[df_all.duplicated(subset=['cloud', 'instance_type'], keep=False)]
if len(duplicates) > 0:
    print(f"⚠️ Found {len(duplicates)} duplicate rows:")
    print(duplicates[['cloud', 'instance_type', 'dbu_rate']].to_string())
df_all = df_all.drop_duplicates(subset=['cloud', 'instance_type'], keep='first')
print(f"✅ Deduplicated: {before_count} → {len(df_all)} rows (removed {before_count - len(df_all)} duplicates)")

# Ensure correct column order
df_final = df_all[['cloud', 'instance_type', 'instance_family', 'vcpus', 'memory_gb', 'dbu_rate', 'is_active', 'source', 'updated_at']].copy()

print(f"📊 Total instance types: {len(df_final)}")
print(f"   - AWS: {len(df_final[df_final['cloud'] == 'AWS'])}")
print(f"   - Azure: {len(df_final[df_final['cloud'] == 'AZURE'])}")
print(f"   - GCP: {len(df_final[df_final['cloud'] == 'GCP'])}")

print(f"\n📊 Instance Families (standardized):")
print(df_final.groupby(['cloud', 'instance_family']).size().unstack(fill_value=0))

df_final.head(20)


In [ ]:
# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema {CATALOG}.{SCHEMA} ready")


In [ ]:
# Convert to Spark DataFrame and save to Delta table
spark_df = spark.createDataFrame(df_final)

# Write to Delta table (overwrite mode for initial load)
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_INSTANCE_RATES)

print(f"✅ Saved {len(df_final)} rows to {TABLE_INSTANCE_RATES}")


In [ ]:
# Verify the data was loaded
display(spark.sql(f"""
    SELECT cloud, instance_family, COUNT(*) as count, 
           ROUND(MIN(dbu_rate), 2) as min_dbu, 
           ROUND(MAX(dbu_rate), 2) as max_dbu
    FROM {TABLE_INSTANCE_RATES}
    GROUP BY cloud, instance_family
    ORDER BY cloud, instance_family
"""))

# Show sample data
display(spark.sql(f"SELECT * FROM {TABLE_INSTANCE_RATES} ORDER BY cloud, instance_family, instance_type LIMIT 20"))
